# RAG Pipeline — Live Walkthrough

This notebook runs the **real** Historical RAG pipeline against an idea card. No mocks. Every cell executes against:

* Azure AI Search — value-stream catalog (`VALUE_STREAM_AZURE_SEARCH_INDEX_NAME`)
* Azure AI Search — historical summary index (`HISTORICAL_AZURE_SEARCH_INDEX_NAME`) OR local FAISS at `HISTORICAL_FAISS_DIR`
* The IDP LLM gateway (condense + Review-Pool calls)
* The Azure embedding service (called internally by the hybrid search)

It does **not** write anywhere — purely read + reason.

Companion reference: [docs/rag.md](rag.md).

## Stages

1. Setup — imports, settings, runtime config
2. Resolve the idea card — from `idea_cards/<TICKET_ID>.*` or pasted text
3. Query prep — `clean_ppt_text`, condense, `_semantic_search_query`
4. Semantic retrieval — value-stream catalog
5. Historical retrieval — analog tickets → per-VS support
6. Merge — lanes, gates, sort, window-fill
7. Review-Pool LLM — final selection + safe-backfill + missed-strong audit
8. Optional — theme payloads + stage predictions
9. End-to-end check — `select_value_streams` in one shot

## 1. Setup

In [ ]:
# Prereqs:
#   * Run from the repo root so `src/` is importable.
#   * Standard project credentials in env (.env): IDP_*, AZURE_*, etc.
#   * For backend='azure' (default) the catalog and historical indexes must be
#     populated. For backend='faiss' you need a built ticket_data/_faiss dir.

from __future__ import annotations

import json
import logging
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from pprint import pprint

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'vs_app').exists():
    REPO = REPO.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('azure').setLevel(logging.WARNING)

print('repo root:', REPO)

In [ ]:
# The knobs you change.
TICKET_ID = 'IDMT-19761'                       # used for source-ticket exclusion + (optional) idea-card lookup
IDEA_CARDS_DIR = REPO / 'idea_cards'           # set per your local layout; default matches the route
REQUESTED_FINAL_OUTPUT = 12                    # max value streams to return
HISTORICAL_BACKEND = os.environ.get('HISTORICAL_SEARCH_BACKEND', 'azure')  # 'azure' or 'faiss'
HISTORICAL_FAISS_DIR = REPO / 'ticket_data' / '_faiss'
HISTORICAL_AZURE_INDEX = os.environ.get('HISTORICAL_AZURE_SEARCH_INDEX_NAME', 'idp_idmt_data')
EXCLUDE_SOURCE_TICKET = True                   # don't let the ticket vote for itself

# Paste the idea-card text directly if you don't have a local file:
IDEA_CARD_TEXT_OVERRIDE = None

In [ ]:
# Real runtime config — same one the pipeline derives at request time.
from vs_app.modules.rag.config.runtime import derive_rag_runtime_config

rt = derive_rag_runtime_config(REQUESTED_FINAL_OUTPUT)
print(f'final_output_count               = {rt.final_output_count}')
print(f'semantic_fetch_k                 = {rt.semantic_fetch_k}')
print(f'historical_ticket_fetch_k        = {rt.historical_ticket_fetch_k}')
print(f'llm_candidate_window             = {rt.llm_candidate_window}')
print(f'max_semantic_plus_historical     = {rt.max_semantic_plus_historical}')
print(f'max_historical_only              = {rt.max_historical_only}')
print(f'max_semantic_only                = {rt.max_semantic_only}')
print(f'max_supporting_tickets_per_cand. = {rt.max_supporting_tickets_per_candidate}')
print()
print(f'idea_card_prompt_chars           = {rt.idea_card_prompt_chars}')
print(f'candidate_description_chars      = {rt.candidate_description_chars}')
print(f'analogs_per_candidate            = {rt.analogs_per_candidate}')
print(f'analog_chars                     = {rt.analog_chars}')
print(f'historical_ticket_ids_per_cand.  = {rt.historical_ticket_ids_per_candidate}')

# Pipeline-level guards on top of runtime config (pipeline.py:43-44)
top_k_semantic   = min(max(1, rt.semantic_fetch_k), 50)
max_ticket_hits  = min(max(1, rt.historical_ticket_fetch_k), 40)
print(f'\nclamped top_k_semantic           = {top_k_semantic}')
print(f'clamped max_ticket_hits          = {max_ticket_hits}')

## 2. Resolve the idea-card text

Same logic the route uses: look for `{TICKET_ID}.{pptx|pdf|docx|txt|md}` under `IDEA_CARDS_DIR`. If nothing is found, fall back to `IDEA_CARD_TEXT_OVERRIDE`.

In [ ]:
from vs_app.integrations.files import idea_card_extractor

raw_idea_card = None
load_source = None

if IDEA_CARD_TEXT_OVERRIDE:
    raw_idea_card = IDEA_CARD_TEXT_OVERRIDE
    load_source = 'IDEA_CARD_TEXT_OVERRIDE'
else:
    try:
        raw_idea_card = idea_card_extractor.extract_idea_card_text(
            doc_id=TICKET_ID,
            local_card_dir=IDEA_CARDS_DIR,
        )
        load_source = f'local file under {IDEA_CARDS_DIR}'
    except FileNotFoundError as exc:
        print(f'No local file found ({exc}). Set IDEA_CARD_TEXT_OVERRIDE and re-run this cell.')
        raise

print(f'idea-card source : {load_source}')
print(f'raw length       : {len(raw_idea_card):,} chars\n')
print('--- first 800 chars ---')
print(raw_idea_card[:800])

## 3. Query preparation

Three transformations (real functions, called in this order by `select_value_streams`):

In [ ]:
# 3.1  clean_ppt_text — strip OCR/unicode noise via shared/text_cleaning.py
from vs_app.modules.rag.query.views import clean_ppt_text

cleaned_query = clean_ppt_text(raw_idea_card)
print(f'raw     : {len(raw_idea_card):,} chars')
print(f'cleaned : {len(cleaned_query):,} chars')
print()
print('--- cleaned (first 800) ---')
print(cleaned_query[:800])

In [ ]:
# 3.2  condense_idea_card — short-circuits if cleaned text <= 3500 chars,
#       otherwise calls gpt-5-mini-idp with the retrieval_summary.yaml prompt.
from vs_app.modules.rag.query.views import condense_idea_card

MAX_CHARS = 3500
needs_llm = len(cleaned_query) > MAX_CHARS
print(f'cleaned length = {len(cleaned_query):,}, threshold = {MAX_CHARS}, LLM call? {needs_llm}\n')

query_for_prompt = condense_idea_card(raw_idea_card, max_chars=MAX_CHARS)
retrieval_query = query_for_prompt or cleaned_query

print(f'query_for_prompt length : {len(query_for_prompt):,} chars')
print(f'retrieval_query length  : {len(retrieval_query):,} chars\n')
print('--- query_for_prompt ---')
print(query_for_prompt)

In [ ]:
# 3.3  Semantic-only preprocessing — normalize + token dedupe (cap 90 unique terms)
from vs_app.modules.rag.query.views import normalize_for_search
from vs_app.modules.rag.retrieval.semantic_retriever import _semantic_search_query

normalized = normalize_for_search(retrieval_query)
semantic_query = _semantic_search_query(retrieval_query)

print(f'normalized      : {len(normalized):,} chars')
print(f'semantic_query  : {len(semantic_query):,} chars ({len(semantic_query.split())} tokens)\n')
print('--- semantic_query (first 500) ---')
print(semantic_query[:500])

## 4. Semantic retrieval — value-stream catalog

Real call to Azure AI Search. Hybrid (BM25 + vector + semantic reranker) over `entity_name` + `content`, filtered to `node_type eq 'ValueStream'`. Top-k clamped to 50 by the pipeline.

In [ ]:
from vs_app.modules.rag.retrieval.semantic_retriever import retrieve_semantic_candidates

semantic_candidates = retrieve_semantic_candidates(
    retrieval_query,
    top_k=top_k_semantic,
    allowed_value_stream_names=None,
)

print(f'{len(semantic_candidates)} semantic candidates returned (after dedupe + sort)\n')
for row in semantic_candidates[:15]:
    print(f"  {row['semantic_score']:>6.4f}  {row['entity_name']}")
if len(semantic_candidates) > 15:
    print(f'  ... and {len(semantic_candidates) - 15} more')

## 5. Historical retrieval — analog tickets

Real call to `idp_idmt_data` (Azure) or `ticket_data/_faiss` (FAISS). Returns both raw ticket hits and an already-grouped per-value-stream support list.

Source-ticket exclusion: when `EXCLUDE_SOURCE_TICKET=True`, the source ticket is removed from the hits so it can't vote for itself.

In [ ]:
from vs_app.modules.rag.retrieval.historical_retriever import (
    filter_historical_result,
    retrieve_historical_support,
)

exclude_ids = [TICKET_ID] if EXCLUDE_SOURCE_TICKET else None

historical = retrieve_historical_support(
    retrieval_query,
    historical_faiss_dir=str(HISTORICAL_FAISS_DIR),
    historical_search_backend=HISTORICAL_BACKEND,
    historical_azure_index_name=HISTORICAL_AZURE_INDEX,
    max_ticket_hits=max_ticket_hits,
    exclude_ticket_ids=exclude_ids,
)
# Belt-and-braces: rebuild support from filtered hits in case the retriever did not
# (legacy paths). select_value_streams does this exact line.
historical = filter_historical_result(historical, exclude_ids)

historical_ticket_hits = historical.get('historical_ticket_hits', [])
historical_vs_support = historical.get('historical_value_stream_support', [])

print(f"historical_source      : {historical.get('historical_source')}")
print(f'historical_ticket_hits : {len(historical_ticket_hits)}')
print(f'value-streams covered  : {len(historical_vs_support)}\n')

print('Top 10 ticket hits:')
for hit in historical_ticket_hits[:10]:
    print(
        f"  {hit.get('best_score', 0):>6.3f}  {hit.get('ticket_id'):14}  "
        f"direct={hit.get('direct_vs_names', [])[:3]}  implied={hit.get('implied_vs_names', [])[:3]}"
    )

In [ ]:
# Per-VS aggregates (already sorted by weighted_support, weighted_direct, best_support)
print(f'{"weighted":>8}  {"hits":>4}  {"direct":>6}  {"impl":>4}  {"best":>5}  {"avg":>5}   value stream')
print('-' * 90)
for row in historical_vs_support[:15]:
    print(
        f"{row['weighted_support_count']:>8.3f}  {row['support_count']:>4}  "
        f"{row['direct_count']:>6}  {row['implied_count']:>4}  "
        f"{row['best_support_score']:>5.3f}  {row['avg_support_score']:>5.3f}   {row['entity_name']}"
    )
if len(historical_vs_support) > 15:
    print(f'... and {len(historical_vs_support) - 15} more')

## 6. Merge — lanes, gates, sort, window-fill

Real `merge_candidate_sources` with the runtime-derived policy. Output includes:

* `merged_candidates` — every union row with `candidate_status` tagged (`sent_to_llm` / `outside_llm_window`)
* `llm_candidates` — the window the LLM will see
* `candidate_window_counts` — lane breakdown of the window

In [ ]:
from vs_app.modules.rag.augmentation.candidate_merger import (
    CandidateWindowPolicy,
    merge_candidate_sources,
)

policy = CandidateWindowPolicy(
    max_semantic_plus_historical=rt.max_semantic_plus_historical,
    max_semantic_only=rt.max_semantic_only,
    max_historical_only=rt.max_historical_only,
    max_supporting_tickets_per_candidate=rt.max_supporting_tickets_per_candidate,
)

augmented = merge_candidate_sources(
    semantic_candidates,
    historical_vs_support,
    policy=policy,
    max_llm_candidates=rt.llm_candidate_window,
)

merged_candidates = augmented['merged_candidates']
llm_candidates = augmented['llm_candidates']

print(f'merged_candidates   : {len(merged_candidates)}')
print(f'llm_candidates      : {len(llm_candidates)} / window={rt.llm_candidate_window}')
print(f'lane counts (window): {augmented["candidate_window_counts"]}\n')

from collections import Counter
print('lane counts (all merged): ', dict(Counter(r['lane'] for r in merged_candidates)))

In [ ]:
# Show the actual LLM window, in the order the LLM will see it
print(f"{'#':>3}  {'lane':28} sem    hits  best   weighted   value stream")
print('-' * 110)
for i, row in enumerate(llm_candidates, start=1):
    print(
        f"{i:>3}  {row['lane']:28} "
        f"{row.get('semantic_score', 0):5.3f}  {row.get('support_count', 0):>4}  "
        f"{row.get('best_support_score', 0):5.3f}  {row.get('weighted_support', row.get('weighted_support_count', 0)):>8.3f}   "
        f"{row['entity_name']}"
    )

In [ ]:
# Optionally inspect candidates that were dropped at lane caps — useful for tuning
dropped = [
    r for r in merged_candidates
    if r.get('candidate_status') == 'outside_llm_window'
]
print(f'{len(dropped)} candidates dropped outside the LLM window (lane caps):\n')
for row in dropped[:15]:
    print(
        f"  [{row['lane']:24}] sem={row.get('semantic_score', 0):5.3f}  "
        f"hits={row.get('support_count', 0):>3}  best={row.get('best_support_score', 0):5.3f}  "
        f"-> {row['entity_name']}"
    )

## 7. Review-Pool LLM call

Real `generate_review_pool_value_streams`. Builds the candidate-block prompt, calls the LLM via `GenerationService.generate_structured`, post-processes (drop unknown ids, safe-backfill, missed-strong audit, reason rewriting).

In [ ]:
from vs_app.modules.rag.augmentation.finalizer import generate_review_pool_value_streams

generated = generate_review_pool_value_streams(
    query_for_prompt=retrieval_query,
    llm_candidates=llm_candidates,
    final_output_count=rt.final_output_count,
    prompt_budget={
        'idea_card_prompt_chars': rt.idea_card_prompt_chars,
        'candidate_description_chars': rt.candidate_description_chars,
        'analogs_per_candidate': rt.analogs_per_candidate,
        'analog_chars': rt.analog_chars,
        'historical_ticket_ids_per_candidate': rt.historical_ticket_ids_per_candidate,
    },
)

selected = generated['selected_value_streams']
raw_response = generated['raw_response']
prompt_debug = raw_response.get('prompt_debug', {}) if isinstance(raw_response, dict) else {}
timing = raw_response.get('timing_ms', {}) if isinstance(raw_response, dict) else {}

print(f'prompt_chars       : {prompt_debug.get("prompt_chars")}')
print(f'system_prompt_chars: {prompt_debug.get("system_prompt_chars")}')
print(f'idea_card_chars    : {prompt_debug.get("idea_card_chars")}')
print(f'candidate_count    : {prompt_debug.get("candidate_count")}')
print(f'final_llm_ms       : {timing.get("final_llm")}')
print(f'total_ms           : {timing.get("total")}')
print()
print(f'{len(selected)} selected value streams:\n')
for row in selected:
    print(f"  conf={row['confidence']:.2f}  [{row.get('selection_source', '?'):14}] {row['entity_name']}")
    print(f"            reason: {row['reason']}")

In [ ]:
# Missed-strong-candidate audit (the model passed on these despite strong evidence)
missed = raw_response.get('missed_strong_candidates', []) if isinstance(raw_response, dict) else []

if not missed:
    print('No missed-strong candidates.')
else:
    print(f'{len(missed)} missed-strong candidate(s):\n')
    for row in missed:
        print(
            f"  [{row.get('lane', '?'):24}] sem={row.get('semantic_score', 0):.3f}  "
            f"hits={row.get('supporting_ticket_count', 0)}  "
            f"best={row.get('best_support_score', 0):.3f}  -> {row.get('entity_name')}"
        )

## 8. Optional — themes + stage predictions

The route also builds Jira-style child theme titles (`{prefix} - {Value Stream}`) and optionally runs the stage-prediction pipeline. Both are independent and can be turned on case by case.

In [ ]:
# 8.1 Theme payloads — derive a child-theme title prefix from the source ticket
from vs_app.modules.themes.title_builder import build_theme_payloads, resolve_theme_title_prefix
from vs_app.modules.rag.query.views import extract_source_ticket_title

source_title = extract_source_ticket_title(raw_idea_card, fallback=TICKET_ID)
theme_prefix, theme_prefix_source = resolve_theme_title_prefix(source_title, None)

theme_payloads = build_theme_payloads(
    source_ticket_id=TICKET_ID,
    source_ticket_title=source_title,
    selected_value_streams=selected,
    theme_title_prefix=theme_prefix,
    theme_title_prefix_source=theme_prefix_source,
)

print(f'source_ticket_title    : {source_title}')
print(f'theme_title_prefix     : {theme_prefix}')
print(f'theme_title_prefix_src : {theme_prefix_source}\n')
for p in theme_payloads[:10]:
    print(f"  - {p.get('theme_title') or p.get('title')}")

In [ ]:
# 8.2 Stage prediction — optional. Set INCLUDE_STAGES = True to call the real predictor.
INCLUDE_STAGES = False

if INCLUDE_STAGES and selected:
    from vs_app.modules.stages.pipeline import predict_stages
    from vs_app.modules.themes.title_builder import enrich_stage_predictions_with_titles

    catalog_index = os.environ.get(
        'VALUE_STREAM_AZURE_SEARCH_INDEX_NAME',
        os.environ.get('AZURE_SEARCH_INDEX_NAME', 'value-streams'),
    )
    stage_result = predict_stages(
        condensed_idea_card=retrieval_query,
        selected_value_streams=selected,
        index_name=catalog_index,
    )
    stage_predictions = enrich_stage_predictions_with_titles(
        stage_predictions=stage_result.get('stage_predictions', []),
        theme_payloads=theme_payloads,
    )
    print(f'{len(stage_predictions)} stage predictions:\n')
    for sp in stage_predictions[:10]:
        pprint(sp)
        print()
else:
    print('Skipped (INCLUDE_STAGES=False or no selected value streams).')

## 9. End-to-end check — single `select_value_streams` call

This is what the FastAPI route actually invokes. We pass the **raw** idea-card text and the same backend settings — every preceding stage will run again inside the pipeline, and the result should match what we built step-by-step above.

In [ ]:
from vs_app.modules.rag.pipeline import select_value_streams

result = select_value_streams(
    raw_idea_card,
    semantic_fetch_k=rt.semantic_fetch_k,
    historical_ticket_fetch_k=rt.historical_ticket_fetch_k,
    llm_candidate_window=rt.llm_candidate_window,
    final_output_count=rt.final_output_count,
    historical_faiss_dir=str(HISTORICAL_FAISS_DIR),
    historical_search_backend=HISTORICAL_BACKEND,
    historical_azure_index_name=HISTORICAL_AZURE_INDEX,
    exclude_ticket_ids=([TICKET_ID] if EXCLUDE_SOURCE_TICKET else None),
)

selected_e2e = result['selected_value_streams']
print(f'{len(selected_e2e)} selected (end-to-end):\n')
for row in selected_e2e:
    print(f"  conf={row['confidence']:.2f}  [{row.get('selection_source', '?'):14}] {row['entity_name']}")

print('\nLane counts in window :', result.get('candidate_window_counts'))
print('Historical source     :', result.get('historical_source'))
print('Excluded ticket IDs   :', result.get('historical_excluded_ticket_ids'))
print('Debug fingerprints    :')
pprint(result['debug']['fingerprints'])

In [ ]:
# Cross-check: did the step-by-step run match the single-call run?
step_by_step = sorted(r['entity_name'] for r in selected)
end_to_end   = sorted(r['entity_name'] for r in selected_e2e)

print(f'step-by-step selected ({len(step_by_step)}): {step_by_step}')
print(f'end-to-end   selected ({len(end_to_end)}): {end_to_end}')
print()
if step_by_step == end_to_end:
    print('MATCH — both paths agree.')
else:
    only_step = set(step_by_step) - set(end_to_end)
    only_e2e  = set(end_to_end) - set(step_by_step)
    print('DIFFER (expected when the LLM is non-deterministic):')
    if only_step: print(f'  only step-by-step: {sorted(only_step)}')
    if only_e2e:  print(f'  only end-to-end:   {sorted(only_e2e)}')

## What you just ran

| Stage | Real function | Output |
| ----- | ------------- | ------ |
| Idea card | `idea_card_extractor.extract_idea_card_text` | raw text |
| Clean | `clean_ppt_text` | normalized text |
| Condense | `condense_idea_card` (LLM if > 3500 chars) | structured-summary blob |
| Semantic | `retrieve_semantic_candidates` | ≤50 VS catalog rows |
| Historical | `retrieve_historical_support` + `filter_historical_result` | ticket hits + per-VS support |
| Merge | `merge_candidate_sources` (with `CandidateWindowPolicy`) | merged candidates + LLM window |
| LLM | `generate_review_pool_value_streams` | picks + missed-strong audit |
| Themes | `build_theme_payloads` | Jira child-theme titles |
| Stages | `predict_stages` (optional) | per-stream stage predictions |
| End-to-end | `select_value_streams` | full payload, route uses this |

### Things to try locally

* Switch `HISTORICAL_BACKEND` to `'faiss'` to read from `ticket_data/_faiss` instead of Azure.
* Set `EXCLUDE_SOURCE_TICKET = False` and watch the source ticket vote for its own value streams in historical support.
* Change `REQUESTED_FINAL_OUTPUT` to 6 — runtime config recomputes the LLM window and lane caps.
* Inspect `result['llm_candidates']` for a candidate that you expected to be selected but wasn't, then find it in `raw_response['missed_strong_candidates']` to see if the model passed on it deliberately.
* Replace `IDEA_CARD_TEXT_OVERRIDE` with a totally different business problem and re-run to see how the lanes shift.